# Bayesian Optimisation of Optogenetic Protrusion Stimulation (Simulated)

This notebook uses a **simulated microscope** with `ProtrusionCell` to find
the optimal optogenetic stimulation parameters that maximise cell protrusions
(measured as area increase).

**Two steerable parameters (BO optimises these):**
- `stim_position_major` – where along the cell's major axis to stimulate (0 = centre, 1 = edge)
- `stim_position_minor` – width of the stimulation band perpendicular to the major axis (0 = narrow, 1 = full width)

**One non-steerable covariate:**
- `opto_rtk_expression` – random per-cell receptor expression level (higher → more response)

**Objective:** maximise `area_change` (fractional area increase after 10 stimulation frames)

The experiment runs through the **full MDA pipeline**: SimulatedMicroscope → segmentation → feature extraction → tracking.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import time
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Suppress JAX/XLA debug messages – env vars must be set before first jax import
os.environ["JAX_LOG_COMPILES"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import gpax
import jax

# Silence all jax internal loggers (created lazily, so we catch them after import)
jax.config.update("jax_log_compiles", False)
for _name in list(logging.Logger.manager.loggerDict):
    if "jax" in _name or "absl" in _name:
        logging.getLogger(_name).setLevel(logging.WARNING)

from rtm_pymmcore.data_structures import (
    Channel,
    StimTreatment,
    SegmentationMethod,
    Fov,
)
import rtm_pymmcore.utils as utils
from rtm_pymmcore.agents.bo_optimization_gpax import (
    BOptGPAX,
    BO_Parameter,
    BO_Objective,
    BO_Covariate,
)
from rtm_pymmcore.microscope.SimulatedMicroscope import SimulatedMicroscope
from rtm_pymmcore.microscope_simulation.cell_protrusion import ProtrusionCell
from rtm_pymmcore.stimulation.two_axis_percentage_of_cell import StimTwoAxisPercentage
from rtm_pymmcore.segmentation.base_segmentation import SegmentatorBinary
from rtm_pymmcore.feature_extraction.simple_fe import SimpleFE
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.img_processing_pip import ImageProcessingPipeline
from scipy.spatial import cKDTree

## Experimental settings

In [3]:
# ---------------------------------------------------------------------------
# Simulated microscope with ProtrusionCells
# ---------------------------------------------------------------------------
mic = SimulatedMicroscope(
    nb_cells=20,
    cell_type="protrusion",
    viewport_width=512,
    viewport_height=512,
    rng_seed=42,
    brownian_d=0.0,
    channel_rendering_modes={0: 1},  # single channel: nucleus fluorescence
    time_scale=3.0,  # speed up physics (default 0.3) so protrusions develop faster
)
mic.mmc.setChannelGroup("Channel")

# ---------------------------------------------------------------------------
# Experiment timing
# ---------------------------------------------------------------------------
FIRST_FRAME_STIMULATION = 2
N_FRAMES = 12  # 2 baseline + 10 stimulation frames
TIME_BETWEEN_TIMESTEPS = 2  # seconds (wall-clock wait between frames)
TIME_PER_FOV = 1  # irrelevant for simulation, needed for real microscopes

# ---------------------------------------------------------------------------
# Storage
# ---------------------------------------------------------------------------
base_path = os.path.join(os.getcwd(), "simulation_output")
experiment_name = "sim_bo_protrusion"
path = os.path.join(base_path, experiment_name)

# ---------------------------------------------------------------------------
# Channels
# ---------------------------------------------------------------------------
channels = [Channel(name="Nucleus", exposure=300)]
condition = ["optoRTK"]

# ---------------------------------------------------------------------------
# Stimulation treatment template (applied at frames 2-11)
# ---------------------------------------------------------------------------
stim_treatment = [
    StimTreatment(
        treatment_name="Protrusion Stim",
        stim_timestep=tuple(range(FIRST_FRAME_STIMULATION, N_FRAMES)),
        stim_exposure_list=(100,) * (N_FRAMES - FIRST_FRAME_STIMULATION),
        stim_power=10,
        stim_channel_name="Stim",
        stim_channel_group="Channel",
        stim_channel_device_name="Spectra",
        stim_channel_power_property_name="Cyan_Level",
    )
]

utils.print_stim_exposures_timesteps(stim_treatment)

# ---------------------------------------------------------------------------
# Image processing pipeline (created once, reused across BO iterations)
# ---------------------------------------------------------------------------
pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=[
        SegmentationMethod(
            name="labels",
            segmentation_class=SegmentatorBinary(),
            use_channel=0,
            save_tracked=True,
        )
    ],
    feature_extractor=SimpleFE("labels"),
    tracker=TrackerTrackpy(search_range=50),
    stimulator=StimTwoAxisPercentage(),
)
mic.set_pipeline(pipeline=pipeline)

Pattern Name:  Protrusion Stim
100 at 2
100 at 3
100 at 4
100 at 5
100 at 6
100 at 7
100 at 8
100 at 9
100 at 10
100 at 11

Directory c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore\simulation_output\sim_bo_protrusion\raw already exists
Directory c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore\simulation_output\sim_bo_protrusion\tracks already exists
Directory c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore\simulation_output\sim_bo_protrusion\stim_mask already exists
Directory c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore\simulation_output\sim_bo_protrusion\stim already exists
Directory c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore\simulation_output\sim_bo_protrusion\particles already exists
Directory c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore\simulation_output\sim_bo_protrusion\labels already exists


## BO Agent subclass

Implements the two abstract methods of `BOptGPAX`:
- `_create_df_acquire_for_exp_cycle` – builds the full acquisition dataframe for one BO iteration
- `_run_experiment_and_preprocess_results` – resets the simulation, runs the MDA pipeline, reads tracks, computes area change, and matches opto-RTK expression from the simulation cells

In [4]:
class ProtrusionBOptGPAX(BOptGPAX):
    """BO agent that runs protrusion experiments on a SimulatedMicroscope.

    Each BO iteration is a separate *phase* (like the multi-phase
    approach in ``01_full_FOV_stimulation_ERK_new_API.ipynb``).  The
    same ``Fov`` object is reused so that cell tracking is continuous
    across iterations.
    """

    def __init__(
        self,
        microscope: SimulatedMicroscope,
        storage_path: str,
        channels: list,
        condition: list,
        stim_treatment: list,
        n_frames: int = 12,
        first_frame_stim: int = 2,
        time_between_timesteps: float = 2,
        time_per_fov: float = 1,
        **kwargs,
    ):
        super().__init__(microscope=microscope, **kwargs)
        self.storage_path = storage_path
        self.channels = channels
        self.condition = condition
        self.stim_treatment = stim_treatment
        self.n_frames = n_frames
        self.first_frame_stim = first_frame_stim
        self.time_between_timesteps = time_between_timesteps
        self.time_per_fov = time_per_fov
        self._phase_counter = 0

        # --- Single FOV, reused for continuous tracking ---
        sim = self.microscope.simulator
        cx = (sim.width - sim.viewport_width) / 2
        cy = (sim.height - sim.viewport_height) / 2
        self._fov = Fov(0)
        self._fov.x = cx
        self._fov.y = cy
        self._fov.z = None
        self._fov.name = "centre"

        # --- Let physics settle and record cell data once ---
        for _ in range(30):
            sim.update(dt=0.05)
        self._df_sim_cells = pd.DataFrame(
            [
                {
                    "cx": c.center[0],
                    "cy": c.center[1],
                    "opto_rtk_expression": c.opto_rtk_expression,
                }
                for c in sim._cells
                if isinstance(c, ProtrusionCell)
            ]
        )

    # ------------------------------------------------------------------

    def _create_df_acquire_for_exp_cycle(self, parameters: dict) -> pd.DataFrame:
        """Build a full df_acquire for one BO iteration (= one phase)."""
        phase_id = self._phase_counter

        df_acquire = utils.generate_df_acquire(
            fovs=[self._fov],
            n_frames=self.n_frames,
            time_between_timesteps=self.time_between_timesteps,
            time_per_fov=self.time_per_fov,
            channels=self.channels,
            phase_name=f"BO_iter_{phase_id}",
            phase_id=phase_id,
            condition=self.condition,
        )

        df_acquire = utils.apply_stim_treatments_to_df_acquire(
            df_acquire,
            self.stim_treatment,
            self.condition,
        )

        # Inject the BO parameters so the stimulator can read them
        df_acquire["stim_position_major"] = parameters["stim_position_major"]
        df_acquire["stim_position_minor"] = parameters["stim_position_minor"]

        return df_acquire

    # ------------------------------------------------------------------

    def _run_experiment_and_preprocess_results(
        self, df_acquire: pd.DataFrame
    ) -> pd.DataFrame:
        """Run one phase of the MDA pipeline and compute area change."""
        phase_id = self._phase_counter
        self._phase_counter += 1

        mic = self.microscope

        # --- Run the experiment (one phase) ---
        mic.run_experiment(df_acquire)

        # --- Read tracks for this specific phase ---
        track_file = os.path.join(
            self.storage_path,
            "tracks",
            f"0_phase_{phase_id}_latest.parquet",
        )
        try:
            df_tracks = pd.read_parquet(track_file)
        except Exception as e:
            print(f"Warning: could not read tracks for phase {phase_id}: {e}")
            return pd.DataFrame()

        if df_tracks.empty or "particle" not in df_tracks.columns:
            print(f"Warning: no tracked cells in phase {phase_id}.")
            return pd.DataFrame()

        # --- Compute area change per tracked cell within this phase ---
        stim_major = float(df_acquire["stim_position_major"].iloc[0])
        stim_minor = float(df_acquire["stim_position_minor"].iloc[0])

        results = []
        for particle_id, grp in df_tracks.groupby("particle"):
            grp_sorted = grp.sort_values("timestep")
            if len(grp_sorted) < 3:
                continue

            baseline = grp_sorted[grp_sorted["timestep"] < self.first_frame_stim]
            if baseline.empty:
                baseline_area = grp_sorted["area"].iloc[0]
            else:
                baseline_area = baseline["area"].mean()

            if baseline_area <= 0:
                continue

            final_area = grp_sorted["area"].iloc[-2:].mean()
            area_change = (final_area - baseline_area) / baseline_area

            cell_x = grp_sorted["y"].iloc[0]
            cell_y = grp_sorted["x"].iloc[0]

            results.append(
                {
                    "particle": particle_id,
                    "area_change": area_change,
                    "cell_x_viewport": cell_x,
                    "cell_y_viewport": cell_y,
                }
            )

        if not results:
            print(f"Warning: no valid area measurements in phase {phase_id}.")
            return pd.DataFrame()

        df_results = pd.DataFrame(results)

        # --- Match tracked cells to sim cells for opto_rtk_expression ---
        fov_x = float(df_acquire["fov_x"].iloc[0])
        fov_y = float(df_acquire["fov_y"].iloc[0])
        df_sim = self._df_sim_cells.copy()
        df_sim["vp_x"] = df_sim["cy"] - fov_y
        df_sim["vp_y"] = df_sim["cx"] - fov_x

        sim = mic.simulator
        vw, vh = sim.viewport_width, sim.viewport_height
        visible = (
            (df_sim["vp_x"] >= 0)
            & (df_sim["vp_x"] < vh)
            & (df_sim["vp_y"] >= 0)
            & (df_sim["vp_y"] < vw)
        )
        df_sim_visible = df_sim[visible].reset_index(drop=True)

        if df_sim_visible.empty:
            print("Warning: no simulation cells visible in viewport.")
            return pd.DataFrame()

        tree = cKDTree(df_sim_visible[["vp_x", "vp_y"]].values)
        tracked_coords = df_results[["cell_x_viewport", "cell_y_viewport"]].values
        _, indices = tree.query(tracked_coords)

        df_results["opto_rtk_expression"] = df_sim_visible.iloc[indices][
            "opto_rtk_expression"
        ].values
        df_results["stim_position_major"] = stim_major
        df_results["stim_position_minor"] = stim_minor

        df_out = df_results[
            [
                "stim_position_major",
                "stim_position_minor",
                "opto_rtk_expression",
                "area_change",
            ]
        ].copy()

        print(
            f"  Phase {phase_id}: {len(df_out)} cells, "
            f"mean area_change={df_out['area_change'].mean():.4f}"
        )
        return df_out

## Configure and run Bayesian Optimisation

In [5]:
bo_params = [
    BO_Parameter(
        name="stim_position_major", bounds=(0.0, 1.0), param_type="float", spacing=0.1
    ),
    BO_Parameter(
        name="stim_position_minor", bounds=(0.1, 1.0), param_type="float", spacing=0.1
    ),
]
bo_covariates = [
    BO_Covariate(name="opto_rtk_expression"),
]
bo_objective = BO_Objective(name="area_change", goal="maximize")


bo_agent = ProtrusionBOptGPAX(
    microscope=mic,
    storage_path=path,
    channels=channels,
    condition=condition,
    stim_treatment=stim_treatment,
    n_frames=N_FRAMES,
    first_frame_stim=FIRST_FRAME_STIMULATION,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    parameters_to_optimize=bo_params,
    objective_metric=bo_objective,
    bo_covariates=bo_covariates,
    n_iterations=12,
    acquisition_function="ucb",
    ucb_beta=4.0,
)

bo_agent.run()

# Finalise the experiment
mic.post_experiment()

=== Initial sample 1/3: {'stim_position_major': np.float32(0.0), 'stim_position_minor': np.float32(1.0)} ===
Total Experiment Time: 0.006111111111111111h
Doing 1 experiment per stim condition


c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore\rtm_pymmcore\stimulation\two_axis_percentage_of_cell.py:110: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  expanded_sub = skimage_binary_dilation(
c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore\rtm_pymmcore\stimulation\two_axis_percentage_of_cell.py:110: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  expanded_sub = skimage_binary_dilation(
c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore\rtm_pymmcore\stimulation\two_axis_percentage_of_cell.py:110: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.

KeyboardInterrupt: 

## Visualise results

In [ ]:
# Collect all measured data from the BO agent's internal state
if bo_agent.x is not None and bo_agent.y is not None:
    x_data = np.array(bo_agent.x)
    y_data = np.array(bo_agent.y).ravel()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: scatter of (stim_position_major, stim_position_minor) coloured by area_change
    sc = axes[0].scatter(
        x_data[:, 0],
        x_data[:, 1],
        c=y_data,
        cmap="viridis",
        s=40,
        edgecolors="k",
        linewidths=0.5,
    )
    axes[0].set_xlabel("stim_position_major")
    axes[0].set_ylabel("stim_position_minor")
    axes[0].set_title("Measured area_change")
    fig.colorbar(sc, ax=axes[0], label="area_change")

    # Right: convergence – best objective seen so far vs iteration
    cumulative_best = np.maximum.accumulate(y_data)
    axes[1].plot(cumulative_best, "o-")
    axes[1].set_xlabel("Measurement index")
    axes[1].set_ylabel("Best area_change so far")
    axes[1].set_title("BO convergence")

    plt.tight_layout()
    plt.show()
else:
    print("No data collected yet.")